In [1]:
import pygame
import random

pygame 2.6.1 (SDL 2.28.4, Python 3.12.3)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [3]:
import pygame
import random
from collections import deque

# ========== Display / Grid ==========
CELL      = 30       # pixel size of a tetris square
COLS      = 10
ROWS      = 20
SIDEBAR_W = 8        # extra columns to show next/hold
WIDTH     = (COLS + SIDEBAR_W) * CELL
HEIGHT    = ROWS * CELL
FPS       = 60

# ========== Colors ==========
BLACK   = (0, 0, 0)
GRAY    = (40, 40, 40)
WHITE   = (255, 255, 255)
GHOST   = (100, 100, 100)
BORDER  = (80, 80, 80)

CYAN    = (0, 255, 255)   # I
YELLOW  = (255, 255, 0)   # O
PURPLE  = (160, 0, 160)   # T
ORANGE  = (255, 165, 0)   # L
BLUE    = (0, 0, 255)     # J
GREEN   = (0, 200, 0)     # S
RED     = (220, 0, 0)     # Z
WWHITE  = (245, 245, 245) # New downward-convex piece

# ========== Shapes ==========
# 1s mark filled cells in the rotation base. We rotate 90° by matrix transform.
# Standard 7 Tetris shapes + REQUIRED NEW white convex shape: [[1,0,1],[1,1,1]]
SHAPES = [
    # I
    {
        "name": "I",
        "matrix": [[1,1,1,1]],
        "color": CYAN
    },
    # O
    {
        "name": "O",
        "matrix": [[1,1],
                   [1,1]],
        "color": YELLOW
    },
    # T
    {
        "name": "T",
        "matrix": [[0,1,0],
                   [1,1,1]],
        "color": PURPLE
    },
    # L
    {
        "name": "L",
        "matrix": [[1,0,0],
                   [1,1,1]],
        "color": ORANGE
    },
    # J
    {
        "name": "J",
        "matrix": [[0,0,1],
                   [1,1,1]],
        "color": BLUE
    },
    # S
    {
        "name": "S",
        "matrix": [[0,1,1],
                   [1,1,0]],
        "color": GREEN
    },
    # Z
    {
        "name": "Z",
        "matrix": [[1,1,0],
                   [0,1,1]],
        "color": RED
    },
    # NEW downward-convex (white): like an inverted arc
    {
        "name": "WCONVEX",
        "matrix": [[1,1,1],
                   [1,0,1]],
        "color": WWHITE
    }
]

def rotate_right(mat):
    # Rotate matrix 90° clockwise: list(zip(*mat[::-1]))
    return [list(row) for row in zip(*mat[::-1])]

def matrix_cells(mat):
    """Yield (r,c) positions where mat[r][c] == 1"""
    for r, row in enumerate(mat):
        for c, v in enumerate(row):
            if v: 
                yield r, c


In [4]:
class Piece:
    def __init__(self, shape_def):
        self.name = shape_def["name"]
        self.base = shape_def["matrix"]
        self.color = shape_def["color"]
        self.rot = 0
        self.matrix = [row[:] for row in self.base]
        # Spawn near top; x centered
        self.r = 0
        self.c = COLS // 2 - len(self.matrix[0]) // 2

    def rotated(self):
        m = self.matrix
        return [list(row) for row in zip(*m[::-1])]

    def apply_rotation(self):
        self.matrix = self.rotated()
        self.rot = (self.rot + 1) % 4

    def width(self):
        return len(self.matrix[0])

    def height(self):
        return len(self.matrix)


class Board:
    def __init__(self, rows=ROWS, cols=COLS):
        self.rows = rows
        self.cols = cols
        self.grid = [[None for _ in range(cols)] for _ in range(rows)]

    def inside(self, r, c):
        return 0 <= r < self.rows and 0 <= c < self.cols

    def collision(self, piece, dr=0, dc=0, rotated_matrix=None):
        mat = rotated_matrix if rotated_matrix else piece.matrix
        for rr, cc in matrix_cells(mat):
            R = piece.r + dr + rr
            C = piece.c + dc + cc
            if not self.inside(R, C) or self.grid[R][C] is not None:
                return True
        return False

    def lock(self, piece):
        for rr, cc in matrix_cells(piece.matrix):
            R = piece.r + rr
            C = piece.c + cc
            if 0 <= R < self.rows and 0 <= C < self.cols:
                self.grid[R][C] = piece.color

    def clear_lines(self):
        cleared = 0
        new_grid = [row for row in self.grid if any(v is None for v in row)]
        cleared = self.rows - len(new_grid)
        for _ in range(cleared):
            new_grid.insert(0, [None for _ in range(self.cols)])
        self.grid = new_grid
        return cleared

    def ghost_drop_row(self, piece):
        # Simulate dropping until collision
        r0 = piece.r
        while not self.collision(piece, dr=1):
            piece.r += 1
        ghost_row = piece.r
        piece.r = r0
        return ghost_row


class Game:
    def __init__(self):
        pygame.init()
        # Create the display once. If notebook re-runs, try to reuse.
        self.screen = pygame.display.set_mode((WIDTH, HEIGHT))
        pygame.display.set_caption("Jupyter Tetris (with extra white convex piece)")
        self.clock = pygame.time.Clock()
        self.font = pygame.font.SysFont("consolas", 18)

        self.board = Board()
        self.score = 0
        self.level = 1
        self.lines = 0
        self.drop_interval = 700  # ms; speeds up with level
        self.last_drop = pygame.time.get_ticks()
        self.paused = False
        self.game_over = False

        self.hold_piece = None
        self.hold_used = False

        # 7+1 bag randomizer
        self.queue = deque()
        self._refill_bag()
        self.current = self._spawn()

    def _refill_bag(self):
        bag = SHAPES[:]
        random.shuffle(bag)
        for s in bag:
            self.queue.append(s)

    def _spawn(self):
        if len(self.queue) < 8:
            self._refill_bag()
        p = Piece(self.queue.popleft())
        # If spawn collides => game over
        if self.board.collision(p, dr=0, dc=0):
            self.game_over = True
        return p

    def hard_drop(self):
        if self.current is None: return
        while not self.board.collision(self.current, dr=1):
            self.current.r += 1
        self._lock_and_next()

    def soft_drop(self):
        if self.current is None: return
        if not self.board.collision(self.current, dr=1):
            self.current.r += 1
            self.score += 1  # soft drop point
        else:
            self._lock_and_next()

    def move(self, dc):
        if self.current is None: return
        if not self.board.collision(self.current, dc=dc):
            self.current.c += dc

    def rotate(self):
        if self.current is None: return
        rotated = self.current.rotated()
        # simple wall kick attempts: try shifts -1,0,+1,+2,-2
        for kick in [0, -1, 1, -2, 2]:
            if not self.board.collision(self.current, dc=kick, rotated_matrix=rotated):
                self.current.c += kick
                self.current.matrix = rotated
                self.current.rot = (self.current.rot + 1) % 4
                return

    def hold(self):
        if self.hold_used or self.current is None: 
            return
        swap = self.hold_piece
        self.hold_piece = Piece({"name": self.current.name, "matrix": self.current.base, "color": self.current.color})
        self.hold_used = True
        if swap is None:
            self.current = self._spawn()
        else:
            self.current = Piece({"name": swap.name, "matrix": swap.base, "color": swap.color})

    def _lock_and_next(self):
        self.board.lock(self.current)
        cleared = self.board.clear_lines()
        if cleared:
            self.lines += cleared
            # Scoring roughly Tetris-like
            points = [0, 100, 300, 500, 800][cleared]
            self.score += points * self.level
            # Level every 10 lines
            new_level = 1 + self.lines // 10
            if new_level != self.level:
                self.level = new_level
                self.drop_interval = max(80, 700 - (self.level-1) * 50)
        self.current = self._spawn()
        self.hold_used = False

    def gravity(self):
        if self.paused or self.game_over: 
            return
        now = pygame.time.get_ticks()
        if now - self.last_drop >= self.drop_interval:
            self.last_drop = now
            if self.current and not self.board.collision(self.current, dr=1):
                self.current.r += 1
            else:
                self._lock_and_next()

    def draw_cell(self, x, y, color, ghost=False):
        rect = pygame.Rect(x, y, CELL, CELL)
        pygame.draw.rect(self.screen, color if not ghost else GHOST, rect)
        pygame.draw.rect(self.screen, BORDER, rect, 1)

    def draw_board(self):
        self.screen.fill(BLACK)
        # Board grid
        for r in range(self.board.rows):
            for c in range(self.board.cols):
                color = self.board.grid[r][c]
                if color:
                    self.draw_cell(c*CELL, r*CELL, color)

        # Ghost piece
        if self.current and not self.game_over:
            ghost_r = self.board.ghost_drop_row(self.current)
            for rr, cc in matrix_cells(self.current.matrix):
                x = (self.current.c + cc) * CELL
                y = (ghost_r + rr) * CELL
                self.draw_cell(x, y, self.current.color, ghost=True)

        # Current piece
        if self.current:
            for rr, cc in matrix_cells(self.current.matrix):
                x = (self.current.c + cc) * CELL
                y = (self.current.r + rr) * CELL
                self.draw_cell(x, y, self.current.color)

        # Sidebar (next / hold / stats)
        sx = COLS * CELL + 10
        sy = 10

        # Next queue (show 4)
        text = self.font.render("NEXT", True, WHITE)
        self.screen.blit(text, (sx, sy))
        ny = sy + 20
        shown = list(self.queue)[:4]
        for i, sdef in enumerate(shown):
            self._draw_mini(sdef["matrix"], sdef["color"], sx, ny + i*80)

        # Hold
        hy = sy + 380
        text = self.font.render("HOLD", True, WHITE)
        self.screen.blit(text, (sx, hy))
        if self.hold_piece:
            self._draw_mini(self.hold_piece.base, self.hold_piece.color, sx, hy+20)

        # Stats
        stats_y = hy + 160
        for label in [f"SCORE: {self.score}", f"LEVEL: {self.level}", f"LINES: {self.lines}"]:
            t = self.font.render(label, True, WHITE)
            self.screen.blit(t, (sx, stats_y))
            stats_y += 22

        if self.paused:
            t = self.font.render("PAUSED (P)", True, WHITE)
            self.screen.blit(t, (sx, stats_y+10))
        if self.game_over:
            t = self.font.render("GAME OVER (R to restart)", True, WHITE)
            self.screen.blit(t, (sx-120, stats_y+40))

    def _draw_mini(self, mat, color, x, y):
        # draw a small preview centered roughly in 4x3 cell area
        scale = CELL // 2
        # get bounds
        h = len(mat)
        w = len(mat[0])
        offset_x = x + 20 - (w*scale)//2
        offset_y = y + 20 - (h*scale)//2
        for rr, cc in matrix_cells(mat):
            rect = pygame.Rect(offset_x + cc*scale, offset_y + rr*scale, scale, scale)
            pygame.draw.rect(self.screen, color, rect)
            pygame.draw.rect(self.screen, BORDER, rect, 1)

    def handle_events(self):
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                return False
            if event.type == pygame.KEYDOWN:
                if event.key in (pygame.K_q, pygame.K_ESCAPE):
                    return False
                if self.game_over:
                    if event.key == pygame.K_r:
                        self.__init__()  # reset game
                    continue
                if event.key == pygame.K_LEFT:
                    self.move(-1)
                elif event.key == pygame.K_RIGHT:
                    self.move(1)
                elif event.key == pygame.K_DOWN:
                    self.soft_drop()
                elif event.key in (pygame.K_UP, pygame.K_x):
                    self.rotate()
                elif event.key == pygame.K_z:
                    # rotate counter-clockwise (3 times CW)
                    for _ in range(3): self.rotate()
                elif event.key == pygame.K_SPACE:
                    self.hard_drop()
                elif event.key == pygame.K_c:
                    self.hold()
                elif event.key == pygame.K_p:
                    self.paused = not self.paused
                elif event.key == pygame.K_r:
                    self.__init__()
        return True

    def step(self):
        """Advance one frame. Returns False if user requested quit."""
        if not pygame.get_init():  # safety if display closed
            return False
        self.clock.tick(FPS)
        keep_running = self.handle_events()
        if not keep_running:
            return False
        self.gravity()
        self.draw_board()
        pygame.display.flip()
        return True


In [ ]:
# Create a single Game instance and run frames in a cell-controlled loop.
# Re-run this cell to resume after a stop; press Q/Esc to exit the loop cleanly.

try:
    game
except NameError:
    game = Game()

running = True
while running:
    running = game.step()
